# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature Vector

For this project, I use a compact set of public-safe content and search-performance features that are available before prediction time.

The selected features are:

- search_volume
- competition
- cpc
- word_count
- char_count
- content_age_days
- days_since_last_update
- ctr
- avg_position

These features are used to predict the target variable `trend_direction`.

In [1]:
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

target = "trend_direction"

print("Selected features:")
for feature in features:
    print("-", feature)

print("\nTarget:", target)
print("\nNumber of features:", len(features))

Selected features:
- search_volume
- competition
- cpc
- word_count
- char_count
- content_age_days
- days_since_last_update
- ctr
- avg_position

Target: trend_direction

Number of features: 9


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
# ML-05 — Section 2: Feature notes

import pandas as pd

feature_notes = pd.DataFrame([
    {
        "feature": "search_volume",
        "meaning": "Estimated search demand for the query/topic.",
        "missing_handling": "Rows with unusable values are excluded or handled before modeling.",
        "categorical": "No",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "competition",
        "meaning": "Competition level associated with the search term.",
        "missing_handling": "Missing values are handled during preprocessing.",
        "categorical": "No",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "cpc",
        "meaning": "Estimated cost-per-click associated with the search term.",
        "missing_handling": "Missing values are handled during preprocessing.",
        "categorical": "No",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "word_count",
        "meaning": "Number of words in the content.",
        "missing_handling": "Missing/unusable values are handled before modeling.",
        "categorical": "No",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "char_count",
        "meaning": "Number of characters in the content.",
        "missing_handling": "Missing/unusable values are handled before modeling.",
        "categorical": "No",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "content_age_days",
        "meaning": "Age of the content measured in days.",
        "missing_handling": "Missing values are handled during preprocessing.",
        "categorical": "No",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "days_since_last_update",
        "meaning": "Number of days since the content was last updated.",
        "missing_handling": "Missing values are handled during preprocessing.",
        "categorical": "No",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "ctr",
        "meaning": "Observed click-through rate associated with search performance.",
        "missing_handling": "Missing values are handled before modeling.",
        "categorical": "No",
        "available_before_prediction": "Timing must be checked"
    },
    {
        "feature": "avg_position",
        "meaning": "Observed average search-result position.",
        "missing_handling": "Missing values are handled before modeling.",
        "categorical": "No",
        "available_before_prediction": "Timing must be checked"
    }
])

feature_notes

,feature,meaning,missing_handling,categorical,available_before_prediction
0,search_volume,Estimated search demand for the query/topic.,Rows with unusable values are excluded or hand...,No,Yes
1,competition,Competition level associated with the search t...,Missing values are handled during preprocessing.,No,Yes
2,cpc,Estimated cost-per-click associated with the s...,Missing values are handled during preprocessing.,No,Yes
3,word_count,Number of words in the content.,Missing/unusable values are handled before mod...,No,Yes
4,char_count,Number of characters in the content.,Missing/unusable values are handled before mod...,No,Yes
5,content_age_days,Age of the content measured in days.,Missing values are handled during preprocessing.,No,Yes
6,days_since_last_update,Number of days since the content was last upda...,Missing values are handled during preprocessing.,No,Yes
7,ctr,Observed click-through rate associated with se...,Missing values are handled before modeling.,No,Timing must be checked
8,avg_position,Observed average search-result position.,Missing values are handled before modeling.,No,Timing must be checked


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# ML-05 — Section 3: Leakage Hunt

import pandas as pd

leakage_audit = pd.DataFrame([
    ["search_volume", "Low",
     "Search-demand signal; safe if measured before the prediction point."],

    ["competition", "Low",
     "Search competition signal available independently of the target label."],

    ["cpc", "Low",
     "Search-market signal; does not directly encode trend_direction."],

    ["word_count", "Low",
     "Content property available before prediction."],

    ["char_count", "Low",
     "Content property available before prediction."],

    ["content_age_days", "Low",
     "Historical content-age feature available at prediction time."],

    ["days_since_last_update", "Low",
     "Historical update-recency feature available at prediction time."],

    ["ctr", "Potential timing risk",
     "Safe only when CTR is calculated from data available before the prediction cutoff."],

    ["avg_position", "Potential timing risk",
     "Safe only when average position uses observations available before the prediction cutoff."]
], columns=["Feature", "Leakage Risk", "Reason"])

print("Leakage Audit")
display(leakage_audit)

print("\nRisk counts:")
print(leakage_audit["Leakage Risk"].value_counts())

Leakage Audit


,Feature,Leakage Risk,Reason
0,search_volume,Low,Search-demand signal; safe if measured before ...
1,competition,Low,Search competition signal available independen...
2,cpc,Low,Search-market signal; does not directly encode...
3,word_count,Low,Content property available before prediction.
4,char_count,Low,Content property available before prediction.
5,content_age_days,Low,Historical content-age feature available at pr...
6,days_since_last_update,Low,Historical update-recency feature available at...
7,ctr,Potential timing risk,Safe only when CTR is calculated from data ava...
8,avg_position,Potential timing risk,Safe only when average position uses observati...



Risk counts:
Leakage Risk
Low                      7
Potential timing risk    2
Name: count, dtype: int64


### Leakage Audit Conclusion

Most selected features have low direct leakage risk because they represent
content properties or search signals that can be available before prediction.

However, `ctr` and `avg_position` require additional timing care. They should
only be used when their values come from an observation window that ends before
the prediction/label window begins.

Therefore, these features are treated as potential timing risks rather than
automatically assuming that they are leakage-free.

No feature that directly contains the target `trend_direction` is included in
the feature vector.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### What I Excluded and Why

I intentionally excluded variables that could create target leakage, expose
private information, or make the model depend on information that would not
be available at prediction time.

The following categories were excluded:

| Excluded data / feature type | Reason |
|---|---|
| `trend_direction` from the feature vector | This is the prediction target and including it as an input would cause direct target leakage. |
| Future-window performance signals | Information collected after the prediction cutoff would leak future information into the model. |
| Direct outcome-derived variables | Variables calculated directly from the target or future outcome could artificially improve model performance. |
| Client names and identifying information | Not required for prediction and excluded for privacy/public-safety reasons. |
| Domains and page URLs | Potentially identifying information and unnecessary for the model feature vector. |
| Private/raw search queries | Excluded to keep the workflow public-safe and avoid exposing sensitive data. |
| Credentials, tokens, and access information | Never included in modeling or public artifacts. |

`client_id` may be used for grouped validation so that the same client does not
appear in both training and testing groups, but it is **not used as a predictive
feature**.

The final predictive feature vector contains only the selected structured
signals documented above. `ctr` and `avg_position` remain subject to timing
checks and should only be used when their observation window occurs before the
prediction/label window.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.